# RetailHero Data Understanding and Exploratory Data Analysis

This notebook examines the raw RetailHero tables before feature engineering
or model training.

The notebook focuses on:

- raw dataset inventory
- table schemas and data types
- table grain and business meaning
- primary-key and relationship checks
- missing values and duplicate records
- client characteristics
- treatment and outcome distributions
- purchase-history structure
- product-data structure
- data-quality issues that affect feature engineering

The competition test set is not used for exploratory analysis, feature
decisions, model training, or validation.

In [1]:
from pathlib import Path
import sys

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

In [2]:
PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "configs" / "data.yaml").exists()
)

SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT

WindowsPath('d:/thao/d/uplif_model/uplif_customer_selection')

In [3]:
from uplift_modeling.data.retailhero import (
    get_retailhero_paths,
    load_retailhero_raw,
)

In [4]:
config_path = PROJECT_ROOT / "configs" / "data.yaml"

with config_path.open("r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

retailhero_config = config["retailhero"]
retailhero_config

{'raw_path': 'data/raw/retailhero',
 'processed_path': 'data/processed/retailhero',
 'files': {'clients': 'clients.csv',
  'products': 'products.csv',
  'purchases': 'purchases.csv',
  'uplift_train': 'uplift_train.csv',
  'uplift_test': 'uplift_test.csv',
  'sample_submission': 'uplift_sample_submission.csv'}}

In [5]:
RAW_DATA_DIR = PROJECT_ROOT / retailhero_config["raw_path"]
PROCESSED_DATA_DIR = PROJECT_ROOT / retailhero_config["processed_path"]

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

data_paths = get_retailhero_paths(RAW_DATA_DIR)

data_paths

{'clients': WindowsPath('d:/thao/d/uplif_model/uplif_customer_selection/data/raw/retailhero/clients.csv'),
 'products': WindowsPath('d:/thao/d/uplif_model/uplif_customer_selection/data/raw/retailhero/products.csv'),
 'purchases': WindowsPath('d:/thao/d/uplif_model/uplif_customer_selection/data/raw/retailhero/purchases.csv'),
 'uplift_train': WindowsPath('d:/thao/d/uplif_model/uplif_customer_selection/data/raw/retailhero/uplift_train.csv'),
 'uplift_test': WindowsPath('d:/thao/d/uplif_model/uplif_customer_selection/data/raw/retailhero/uplift_test.csv'),
 'sample_submission': WindowsPath('d:/thao/d/uplif_model/uplif_customer_selection/data/raw/retailhero/uplift_sample_submission.csv')}

In [6]:
raw_tables = load_retailhero_raw(
    raw_dir=RAW_DATA_DIR,
    nrows=None,
)

clients = raw_tables["clients"]
products = raw_tables["products"]
uplift_train = raw_tables["uplift_train"]
uplift_test = raw_tables["uplift_test"]

clients["first_issue_date"] = pd.to_datetime(
    clients["first_issue_date"],
    errors="coerce",
)

clients["first_redeem_date"] = pd.to_datetime(
    clients["first_redeem_date"],
    errors="coerce",
)

print("clients:", clients.shape)
print("products:", products.shape)
print("uplift_train:", uplift_train.shape)
print("uplift_test:", uplift_test.shape)

clients: (400162, 5)
products: (43038, 11)
uplift_train: (200039, 3)
uplift_test: (200123, 1)


In [7]:
duckdb_connection = duckdb.connect(
    database=":memory:"
)

In [8]:
def to_sql_path(path: Path) -> str:
    """Return a normalized path safe for SQL literals."""
    return (
        path.resolve()
        .as_posix()
        .replace("'", "''")
    )


purchases_csv_path = data_paths["purchases"]
purchases_csv_sql = to_sql_path(purchases_csv_path)